In [ ]:
!pip install spiceypy
!pip install plotly

In [1]:
import datetime
import glob

from matplotlib import pyplot as pyplot
import numpy as np
import pandas as pd
import plotly.graph_objects as go

import spiceypy

C:\Users\Owner\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
# First we need to download some kernels

# Leapseconds
!curl https://naif.jpl.nasa.gov/pub/naif/generic_kernels/lsk/naif0012.tls --create-dirs -o kernels/lsk/naif0012.tls

# SPK
!curl https://naif.jpl.nasa.gov/pub/naif/generic_kernels/spk/planets/de432s.bsp --create-dirs -o kernels/spk/de432s.bsp

# PCK
!curl https://naif.jpl.nasa.gov/pub/naif/generic_kernels/pck/gm_de440.tpc --create-dirs -o kernels/pck/gm_de440.tpc

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  5257  100  5257    0     0   4960      0  0:00:01  0:00:01 --:--:--  4992
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
 19 10.3M   19 2038k    0     0  1867k      0  0:00:05  0:00:01  0:00:04 1936k
100 10.3M  100 10.3M    0     0  6010k      0  0:00:01  0:00:01 --:--:-- 6143k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   

In [3]:
kernel_filepaths = glob.glob("kernels/**/*")
print(kernel_filepaths)

['kernels\\lsk\\naif0012.tls', 'kernels\\pck\\gm_de440.tpc', 'kernels\\spk\\de432s.bsp']


In [4]:
spiceypy.furnsh(kernel_filepaths)

In [5]:
datetime_now = datetime.datetime.today()
datetime_now = datetime_now.strftime("%Y-%m-%dT%H:%M:%S")
print(datetime_now)

2026-09-09T17:33:32


In [7]:
et_now = spiceypy.utc2et(datetime_now)
print(et_now)

842247281.1825043


In [8]:
earth_state_wrt_sun, earth_sun_light_time = spiceypy.spkgeo(targ = 399,
                                                        et = et_now,
                                                        ref="ECLIPJ2000",
                                                        obs=10
                                                      )
print(earth_state_wrt_sun)
print(earth_sun_light_time)

[ 1.46626386e+08 -3.46905818e+07  1.19806227e+03  6.38214959e+00
  2.88875534e+01 -5.99631144e-04]
502.59524073165625


In [9]:
earth_sun_lt_min = earth_sun_light_time / 60
print(f"Light travel time between Earth and Sun: {earth_sun_lt_min}")

Light travel time between Earth and Sun: 8.376587345527604


In [10]:
earth_sun_dist_km = np.linalg.norm(earth_state_wrt_sun[:3])
print(f"Distance between Earth and Sun in km: {earth_sun_dist_km}")

Distance between Earth and Sun in km: 150674262.5980449


In [11]:
# ... pretty unhandy. Let's compute it in Astronomical Units (AU). It should be 1, right???
earth_sun_dist_au = spiceypy.convrt(earth_sun_dist_km, 'km', 'AU')
print(f"Distance between Earth and Sun in AU: {earth_sun_dist_au}")

Distance between Earth and Sun in AU: 1.0071952360012906


# Earth's Orbital Elements
Depending on when you compute the state vector of our home planet your will get a distance between Earth and Sun that is larger or smaller 1! That's because the Earth is not on a "100 %" perfect circle. It is moving in an elliptical orbit. The "elliptic-ness" is defined by the eccentricity e; one of the 6 Keplerian Elements that define the shape and orientation of an orbit (a 7th value determines the position of an object).

Circular Orbit: e Elliptic Orbit: 0 < e < 1 Parabolic Orbit: e = 1 Hyperbolic Orbit: e > 1



In [12]:
# First we need to set the gravitational parameter
_, grav_mu = spiceypy.bodvrd("SUN", "GM", 1)
grav_mu = grav_mu[0]
print(grav_mu)

132712440041.27939


In [13]:
# Computing the keplerian elements
(earth_peri_km,
 earth_ecc,
 earth_incl_rad,
 earth_lnode_asc_rad,
 earth_argp_rad,
 m0_rad,
 t0,
 mu) = spiceypy.oscelt(earth_state_wrt_sun, et_now, grav_mu)

print(f"Earth's eccentricity: {earth_ecc}")

Earth's eccentricity: 0.016167999694180556


In [14]:
start_date = "2025-01-01"
end_date = "2026-01-01"

comp_days = np.arange(start_date, end_date, dtype='datetime64[D]')
comp_days = comp_days.astype(str)

print(comp_days)

['2025-01-01' '2025-01-02' '2025-01-03' '2025-01-04' '2025-01-05'
 '2025-01-06' '2025-01-07' '2025-01-08' '2025-01-09' '2025-01-10'
 '2025-01-11' '2025-01-12' '2025-01-13' '2025-01-14' '2025-01-15'
 '2025-01-16' '2025-01-17' '2025-01-18' '2025-01-19' '2025-01-20'
 '2025-01-21' '2025-01-22' '2025-01-23' '2025-01-24' '2025-01-25'
 '2025-01-26' '2025-01-27' '2025-01-28' '2025-01-29' '2025-01-30'
 '2025-01-31' '2025-02-01' '2025-02-02' '2025-02-03' '2025-02-04'
 '2025-02-05' '2025-02-06' '2025-02-07' '2025-02-08' '2025-02-09'
 '2025-02-10' '2025-02-11' '2025-02-12' '2025-02-13' '2025-02-14'
 '2025-02-15' '2025-02-16' '2025-02-17' '2025-02-18' '2025-02-19'
 '2025-02-20' '2025-02-21' '2025-02-22' '2025-02-23' '2025-02-24'
 '2025-02-25' '2025-02-26' '2025-02-27' '2025-02-28' '2025-03-01'
 '2025-03-02' '2025-03-03' '2025-03-04' '2025-03-05' '2025-03-06'
 '2025-03-07' '2025-03-08' '2025-03-09' '2025-03-10' '2025-03-11'
 '2025-03-12' '2025-03-13' '2025-03-14' '2025-03-15' '2025-03-16'
 '2025-03-

In [15]:
elements_df = pd.DataFrame(comp_days, columns=["date"])

In [16]:
elements_df

,date
0,2025-01-01
1,2025-01-02
2,2025-01-03
3,2025-01-04
4,2025-01-05
...,...
360,2025-12-27
361,2025-12-28
362,2025-12-29
363,2025-12-30
